In [1]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
import os

c:\Users\gaura\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
hf_token = os.getenv("HF_TOKEN")

In [4]:
llm_endpoint = HuggingFaceEndpoint(
    repo_id="deepseek-ai/DeepSeek-R1",
    task = 'text-generation',
    temperature=0.9,
    huggingfacehub_api_token=hf_token)

In [ ]:
model = ChatHuggingFace(llm=llm_endpoint) 

In [7]:
response = model.invoke('Tell me about germany?')

In [8]:
response.content

"\nGermany is a fascinating country in **Central Europe** with a rich history, powerful economy, and vibrant culture. Here's a comprehensive overview:\n\n1.  **Geography:**\n    *   **Location:** Borders 9 countries: Denmark (north), Poland & Czech Republic (east), Austria & Switzerland (south), France, Luxembourg, Belgium & Netherlands (west).\n    *   **Landscape:** Diverse - North German Plain, forested central uplands, Alps in the south (Zugspitze is highest peak at 2,962m). Major rivers include Rhine, Danube, Elbe.\n    *   **Climate:** Temperate seasonal, with moderate summers and cold winters.\n\n2.  **History (Key Highlights):**\n    *   **Early Tribes & Holy Roman Empire:** Home to Germanic tribes; became the core of the Holy Roman Empire (800/962 - 1806).\n    *   **German Confederation & Empire:** Following Napoleon, the German Confederation formed. Otto von Bismarck unified most states into the German Empire in 1871.\n    *   **World Wars:** Key instigator of World War I (1

## String Parser

In [9]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [10]:
templet1 = PromptTemplate(template="Get a detailed report on the following topic: {topic}",
                          input_variables=['topic'])

templte2 = PromptTemplate(template="Create a 5 point summary of the follwoing report {report}",
                          input_variables=['report'])

In [11]:
parser = StrOutputParser()

In [12]:
chain = templet1 | model | parser | templte2 | model | parser

In [13]:
response = chain.invoke({'topic':'LLM vs AI agent'})

In [14]:
print(response)


Here's a concise 5-point summary of the report:

1. **Fundamental Distinction**  
   LLMs are specialized language models for text processing/generation, while AI Agents are autonomous systems that perform tasks by interacting with environments using multiple components.

2. **Core Architecture**  
   LLMs operate as single-component models (e.g., GPT-4, Claude). AI Agents integrate LLMs with tools (APIs, databases), memory systems, planning modules, and feedback loops.

3. **Autonomy & Function**  
   LLMs reactively generate text responses to prompts. AI Agents proactively plan, make decisions, and execute actions (e.g., booking flights, data analysis) autonomously to achieve goals.

4. **Memory & Context**  
   LLMs are stateless (no inherent memory between interactions). AI Agents are stateful, retaining context, goal history, and environmental data for ongoing tasks.

5. **Output & Interaction**  
   LLMs produce text outputs (summaries, translations). AI Agents generate actions:

## JSON Parser

In [15]:
from langchain_core.output_parsers import JsonOutputParser

In [16]:
parser = JsonOutputParser()

In [17]:
templet_new = PromptTemplate(template='''Get the name, country, date of birth
                             and date of death of the following person : {person}
                             In the given format : {output_format}''',
                             input_variables=['person'],
                             partial_variables={'output_format':parser.get_format_instructions()})  

In [18]:
chain = templet_new | model | parser

In [19]:
response = chain.invoke({'person':'Dalai lama'})

In [20]:
response

{'name': 'Tenzin Gyatso',
 'country': 'China',
 'date_of_birth': '1935-07-06',
 'date_of_death': None}

## Structure Output parser

In [24]:
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema

In [25]:
schema = [ResponseSchema(name='Full Name',description='Full and real name of the person'),
          ResponseSchema(name='Country',description='Country where the person was born'),
          ResponseSchema(name='DOB',description='Date of birth of person'),
          ResponseSchema(name='DOD',description='Date of death of person')]

In [27]:
parser = StructuredOutputParser.from_response_schemas(schema)

In [28]:
templet_new = PromptTemplate(template='''Give the description of the following person : {person}
                             In the given format : {output_format}''',
                             input_variables=['person'],
                             partial_variables={'output_format':parser.get_format_instructions()})  

In [29]:
chain = templet_new | model | parser

In [30]:
response = chain.invoke({'person':'Manmohan Singh'})

In [31]:
response

{'Full Name': 'Manmohan Singh',
 'Country': 'British India (present-day Pakistan)',
 'DOB': 'September 26, 1932',
 'DOD': ''}

## Pydantic Parser

In [33]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

In [43]:
class Details(BaseModel):
    country : str = Field(description='Name of the country')
    capital : str = Field(default='Not Available',description='capital of the country')
    currency : str = Field(description='Currency of the country')
    exchange : float = Field(ge=0,description='one unit of this currency equal to how many rupees')


In [44]:
parser = PydanticOutputParser(pydantic_object=Details)

In [45]:
templet_new = PromptTemplate(template='''Give the details of this country : {country}
                             In the given format : {output_format}''',
                             input_variables=['country'],
                             partial_variables={'output_format':parser.get_format_instructions()})  

In [46]:
chain = templet_new | model | parser

In [47]:
response = chain.invoke({'country':'Germany'})

In [48]:
response.model_dump()

{'country': 'Germany',
 'capital': 'Berlin',
 'currency': 'Euro',
 'exchange': 90.0}